# Lecture 3: ARMA Models
- 3.1 ARMA(p,q) Processes
- 3.2 ACF and PACF of ARMA(p,q) Processes
- 3.3 Forecasting ARMA Processes


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima_process import arma_generate_sample
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf
from scipy.linalg import toeplitz
import warnings; warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 4)
print("Ready!")


## 3.1 ARMA(p, q) Processes

**Definition:** $\{X_t\}$ is an ARMA(p,q) process if:
$$X_t - \phi_1 X_{t-1} - \cdots - \phi_p X_{t-p} = Z_t + \theta_1 Z_{t-1} + \cdots + \theta_q Z_{t-q}$$

In operator notation: $\phi(B)X_t = \theta(B)Z_t$

**Causality Condition:** $\phi(z) = 1 - \phi_1 z - \cdots - \phi_p z^p \neq 0$ for $|z| \leq 1$

**Invertibility Condition:** $\theta(z) = 1 + \theta_1 z + \cdots + \theta_q z^q \neq 0$ for $|z| \leq 1$


In [ ]:
# ── Characteristic Roots and Causality/Invertibility Check ──

def check_causality_invertibility(ar_params, ma_params):
    '''ar_params: [phi1, phi2, ...] (signs after the operator)'''
    # AR polynomial roots: phi(z) = 1 - phi1*z - phi2*z^2 - ...
    ar_poly = np.array([1] + [-p for p in ar_params])
    ar_roots = np.roots(ar_poly)
    
    ma_poly = np.array([1] + list(ma_params))
    ma_roots = np.roots(ma_poly)
    
    print(f"AR roots: {ar_roots}")
    print(f"  |AR roots|: {np.abs(ar_roots).round(4)}")
    print(f"  Causal? (all roots |z|>1): {all(np.abs(ar_roots) > 1)}")
    print(f"MA roots: {ma_roots}")
    print(f"  |MA roots|: {np.abs(ma_roots).round(4)}")
    print(f"  Invertible? (all roots |z|>1): {all(np.abs(ma_roots) > 1)}")
    print()
    return ar_roots, ma_roots

models_to_check = [
    ("AR(1) phi=0.8",      [0.8],       []),
    ("AR(1) phi=1.2",      [1.2],       []),     # non-causal!
    ("AR(2) phi1=0.5,phi2=0.3", [0.5,0.3], []),
    ("MA(1) theta=0.6",    [],          [0.6]),
    ("MA(1) theta=1.5",    [],          [1.5]),  # non-invertible!
    ("ARMA(1,1)",          [0.7],       [0.4]),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
theta = np.linspace(0, 2*np.pi, 300)
unit_circle_x = np.cos(theta)
unit_circle_y = np.sin(theta)

for idx, (name, ar, ma) in enumerate(models_to_check):
    ax = axes[idx//3, idx%3]
    ax.plot(unit_circle_x, unit_circle_y, 'k-', lw=1.5, label='Unit circle')
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    
    print(f"{'='*45}")
    print(f"Model: {name}")
    ar_r, ma_r = check_causality_invertibility(ar, ma)
    
    if len(ar_r) > 0:
        ax.scatter(1/ar_r.real, 1/ar_r.imag if np.any(ar_r.imag != 0) else np.zeros(len(ar_r)),
                   marker='x', s=100, color='red', zorder=5, label='AR roots (1/z)')
    if len(ma_r) > 0:
        ax.scatter(1/ma_r.real, 1/ma_r.imag if np.any(ma_r.imag != 0) else np.zeros(len(ma_r)),
                   marker='o', s=100, color='blue', zorder=5, label='MA roots (1/z)')
    
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
    ax.set_aspect('equal'); ax.set_title(name, fontsize=10)
    ax.legend(fontsize=7)
    ax.fill_between(unit_circle_x, unit_circle_y, alpha=0.05, color='gray')

plt.tight_layout(); plt.show()


## 3.2 ACF and PACF Computation

### 3.2.1 Yule-Walker Equations (for AR processes)

For the AR(p) model: $\phi(B)X_t = Z_t$

Yule-Walker equations:
$$\rho(h) - \phi_1\rho(h-1) - \cdots - \phi_p\rho(h-p) = 0, \quad h > 0$$

In matrix form: $\Gamma_p \boldsymbol{\phi} = \boldsymbol{\gamma}_p$

### 3.2.2 General ACF for ARMA

For $|h| > q$: $\phi(B)\rho(h) = 0$, so the ACF decays according to the AR polynomial.


In [ ]:
# ── Theoretical ACF via Yule-Walker ──

def ar_acf_yulewalker(phi_params, max_lag=30):
    '''Compute the theoretical ACF of AR(p) by solving the Yule-Walker equations'''
    p = len(phi_params)
    phi = np.array(phi_params)
    
    from statsmodels.tsa.arima_process import arma_acovf
    ar = np.array([1] + [-p for p in phi_params])
    ma = np.array([1])
    acvf = arma_acovf(ar, ma, nobs=max_lag+1)
    rho = acvf / acvf[0]
    return rho

# Compare different AR models
ar_models = {
    'AR(1) phi=0.7':            [0.7],
    'AR(1) phi=-0.7':           [-0.7],
    'AR(2) phi1=0.7, phi2=-0.2': [0.7, -0.2],
    'AR(2) complex roots':       [1.0, -0.5],
}

fig, axes = plt.subplots(len(ar_models), 2, figsize=(14, 10))
lags = 25

for i, (name, phi) in enumerate(ar_models.items()):
    rho = ar_acf_yulewalker(phi, max_lag=lags)
    
    axes[i, 0].stem(range(lags+1), rho, linefmt='steelblue', 
                     markerfmt='o', basefmt='k-')
    axes[i, 0].axhline(0, color='k', lw=0.5)
    ci = 1.96 / np.sqrt(300)
    axes[i, 0].axhline(ci, color='red', ls='--', lw=0.8, alpha=0.7)
    axes[i, 0].axhline(-ci, color='red', ls='--', lw=0.8, alpha=0.7)
    axes[i, 0].set_title(f'{name} — Theoretical ACF')
    axes[i, 0].set_xlabel('Lag h')
    
    # Simulation
    from statsmodels.tsa.arima_process import arma_generate_sample
    ar_poly = np.array([1] + [-p for p in phi])
    X = arma_generate_sample(ar_poly, [1], 300)
    plot_pacf(X, lags=20, ax=axes[i, 1], color='darkorange', title=f'{name} — Sample PACF')

plt.tight_layout(); plt.show()


## 3.2.4 Model Identification: ACF and PACF Summary

| Model | ACF | PACF |
|-------|-----|------|
| AR(p) | Exponential/sinusoidal decay | Cuts off after lag p |
| MA(q) | Cuts off after lag q | Exponential/sinusoidal decay |
| ARMA(p,q) | Both tail off | Both tail off |


In [ ]:
# ── Model Identification Guide: Interactive Comparison ──
np.random.seed(42)
n = 500

configs = [
    ('AR(1)', [1, -0.8], [1], 'ACF: exponential decay, PACF: cuts off at lag 1'),
    ('AR(2)', [1,-0.6,-0.3],[1], 'ACF: exponential/oscillatory, PACF: cuts off at lag 2'),
    ('MA(1)', [1], [1, 0.7], 'ACF: cuts off at lag 1, PACF: exponential decay'),
    ('MA(2)', [1], [1,-0.5,0.3], 'ACF: cuts off at lag 2, PACF: tails off'),
    ('ARMA(1,1)',[1,-0.7],[1,0.4],'ACF and PACF: both tail off'),
    ('ARMA(2,1)',[1,-0.5,-0.2],[1,0.3],'ACF and PACF: both tail off'),
]

fig, axes = plt.subplots(len(configs), 3, figsize=(16, 18))

for i, (name, ar, ma, desc) in enumerate(configs):
    X = arma_generate_sample(ar, ma, n)
    
    axes[i, 0].plot(X[:100], lw=0.9, color='steelblue')
    axes[i, 0].set_title(f'{name} — {desc}', fontsize=9)
    axes[i, 0].axhline(0, color='gray', ls='--', lw=0.5)
    
    plot_acf(X, lags=20, ax=axes[i, 1], color='steelblue', title='ACF')
    plot_pacf(X, lags=20, ax=axes[i, 2], color='darkorange', title='PACF', method='ywm')

plt.tight_layout(); plt.show()


## 3.3 Forecasting ARMA Processes

The $h$-step forecast for an ARMA(p,q) process:

$$\hat{X}_{n+h} = \phi_1 \hat{X}_{n+h-1} + \cdots + \phi_p \hat{X}_{n+h-p} + \hat{Z}_{n+h} + \theta_1 \hat{Z}_{n+h-1} + \cdots + \theta_q \hat{Z}_{n+h-q}$$

where $\hat{X}_j = X_j$ ($j \leq n$) and $\hat{Z}_j = 0$ ($j > n$), $\hat{Z}_j = X_j - \hat{X}_j$ ($j \leq n$).

**Forecast mean:** as $h \to \infty$, $\hat{X}_{n+h} \to \mu = 0$

**Forecast variance:** $v(h) = \sigma^2 \sum_{j=0}^{h-1} \psi_j^2$ (as $h \to \infty$ converges to $\sigma^2 \sum_{j=0}^{\infty} \psi_j^2 = \gamma(0)$)


In [ ]:
# ── ARMA Forecasts and Confidence Intervals ──
from statsmodels.tsa.arima.model import ARIMA

np.random.seed(10)
n_train = 150
n_forecast = 30

# Generate ARMA(2,1) data
from statsmodels.tsa.arima_process import arma_generate_sample
X_full = arma_generate_sample([1, -0.6, -0.2], [1, 0.4], n_train + n_forecast, scale=1)
X_train = X_full[:n_train]
X_test  = X_full[n_train:]

# Fit model
model = ARIMA(X_train, order=(2, 0, 1))
fit   = model.fit()
print(fit.summary())

# Forecast
forecast = fit.get_forecast(steps=n_forecast)
fc_mean  = forecast.predicted_mean
fc_ci    = forecast.conf_int(alpha=0.05)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: full series + forecast
t_all = np.arange(n_train + n_forecast)
axes[0].plot(t_all[:n_train], X_train, 'steelblue', lw=1.2, label='Training data')
axes[0].plot(t_all[n_train:], X_test, 'green', lw=1.2, label='True values')
axes[0].plot(t_all[n_train:], fc_mean, 'r-', lw=2, label='ARMA(2,1) forecast')
axes[0].fill_between(t_all[n_train:], fc_ci[:, 0], fc_ci[:, 1],
                      alpha=0.2, color='red', label='95% CI')
axes[0].axvline(n_train, color='k', ls='--', lw=1)
axes[0].legend(fontsize=9); axes[0].set_title('ARMA(2,1) h-Step Forecast')

# Right: forecast uncertainty
h_vals = np.arange(1, n_forecast+1)
pred_std = np.array([forecast.se_mean[h] for h in range(n_forecast)])

axes[1].plot(h_vals, pred_std, 'r-o', ms=4, lw=1.5, label='Forecast Std. Error')
axes[1].axhline(np.sqrt(X_train.var()), color='blue', ls='--', label='Process std (gamma(0)^0.5)')
axes[1].set_xlabel('Forecast horizon h'); axes[1].set_ylabel('Standard Error')
axes[1].set_title('Forecast Uncertainty Grows with h')
axes[1].legend()

plt.tight_layout(); plt.show()

# Error metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error
mse = mean_squared_error(X_test, fc_mean)
mae = mean_absolute_error(X_test, fc_mean)
print(f"\nForecast MSE:  {mse:.4f}")
print(f"Forecast MAE:  {mae:.4f}")
print(f"Forecast RMSE: {np.sqrt(mse):.4f}")


In [ ]:
# ── Forecast Horizon and Model Comparison ──
from statsmodels.tsa.arima.model import ARIMA

np.random.seed(7)
n = 200

# AR(1) data
X = arma_generate_sample([1, -0.8], [1], n, scale=1)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col_idx, h in enumerate([1, 5, 20]):
    errors = []
    for start in range(50, n - h):
        model = ARIMA(X[:start], order=(1, 0, 0))
        fit = model.fit()
        fc = fit.forecast(steps=h)
        errors.append(X[start + h - 1] - (fc[-1] if hasattr(fc, "iloc") else fc[-1]))
    
    axes[0, col_idx].hist(errors, bins=25, color='steelblue', alpha=0.7, edgecolor='white')
    axes[0, col_idx].axvline(0, color='red', ls='--')
    axes[0, col_idx].set_title(f'{h}-Step Forecast Errors')
    axes[0, col_idx].set_xlabel('Error')
    
    axes[1, col_idx].plot(errors, lw=0.7, color='steelblue')
    axes[1, col_idx].axhline(0, color='red', ls='--')
    axes[1, col_idx].axhline(np.std(errors), color='green', ls=':', label=f'sigma={np.std(errors):.3f}')
    axes[1, col_idx].axhline(-np.std(errors), color='green', ls=':')
    axes[1, col_idx].set_title(f'{h}-Step Error Over Time'); axes[1, col_idx].legend()
    
    print(f"h={h:2d}: Mean Error={np.mean(errors):.4f}, Std={np.std(errors):.4f}, RMSE={np.sqrt(np.mean(np.array(errors)**2)):.4f}")

plt.tight_layout(); plt.show()
print("\nObservation: forecast uncertainty increases with h!")


## Summary

| Concept | Detail |
|---------|--------|
| **ARMA(p,q)** | $\phi(B)X_t = \theta(B)Z_t$ |
| **Causality** | AR roots lie outside the unit circle |
| **Invertibility** | MA roots lie outside the unit circle |
| **ACF-PACF ID** | AR ↔ PACF cuts off; MA ↔ ACF cuts off |
| **h-step forecast** | Uncertainty grows with h; converges to the mean at long horizons |

